# 02 — Application output regression

This notebook reruns the application-level Yamada cases on both `origin/Latest_Workplace` and the current optimized `HEAD`.

It requires literal equality of the reproduced records, including graph diagnostics, selected projection information where relevant, PD codes, and exact Yamada polynomial strings. `Latest_Workplace` is intentionally retained here as the historical correctness baseline; no other notebook or production path should depend on it.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import tempfile

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src' / 'knotted_graph').exists():
    raise RuntimeError('Run this notebook from inside the KnottedGraph checkout.')
DRIVER = ROOT / 'dev' / 'application_yamada_regression.py'
REF_RUNNER = ROOT / 'dev' / 'run_application_yamada_regression_ref.py'

import knotted_graph
from knotted_graph.invariants.yamada.native import native_available, native_import_error
from knotted_graph.invariants.yamada.factorized_frontier import native_factorized_available
print('Current notebook environment:', sys.executable)
print('Current KnottedGraph:', Path(knotted_graph.__file__).resolve())
print('Native resolved-graph Yamada backend:', native_available())
print('Factorized diagram Yamada backend:', native_factorized_available())
print('Native import error:', native_import_error())
assert native_available(), native_import_error()
assert native_factorized_available()
print('NOTE: this notebook intentionally executes detached source worktrees for correctness comparison only; it is not a performance benchmark.')

subprocess.run(['git', 'fetch', 'origin', 'Latest_Workplace'], cwd=ROOT, check=True, capture_output=True, text=True)

def run_ref(ref):
    with tempfile.TemporaryDirectory() as td:
        work = Path(td) / 'repo'
        add = subprocess.run(['git', 'worktree', 'add', '--detach', str(work), ref], cwd=ROOT, text=True, capture_output=True)
        if add.returncode:
            raise RuntimeError(f'Could not create worktree for {ref}:\n{add.stdout}\n{add.stderr}')
        try:
            env = dict(os.environ)
            env.pop('PYTHONPATH', None)
            env['PYTHONNOUSERSITE'] = '1'
            env['KG_REGRESSION_SRC'] = str(work / 'src')
            env['KG_REGRESSION_DRIVER'] = str(DRIVER)
            proc = subprocess.run([sys.executable, str(REF_RUNNER)], cwd=work, env=env, text=True, capture_output=True)
            if proc.returncode:
                raise RuntimeError(f'Application regression failed for {ref}.\nSTDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}')
            return json.loads(proc.stdout)
        finally:
            subprocess.run(['git', 'worktree', 'remove', '--force', str(work)], cwd=ROOT, text=True, capture_output=True, check=False)

baseline = run_ref('origin/Latest_Workplace')
optimized = run_ref('HEAD')
print('baseline records =', len(baseline))
print('optimized records =', len(optimized))


In [ ]:
if baseline != optimized:
    differences = []
    for index, (old, new) in enumerate(zip(baseline, optimized)):
        if old != new:
            differences.append((index, old, new))
    if len(baseline) != len(optimized):
        print('record counts differ:', len(baseline), len(optimized))
    for index, old, new in differences[:10]:
        print('\nDIFFERENCE', index)
        print('Latest_Workplace:', json.dumps(old, indent=2, sort_keys=True))
        print('YAMADA_Optimization_LATEST:', json.dumps(new, indent=2, sort_keys=True))
    raise AssertionError('Application-level Yamada output changed relative to Latest_Workplace.')

print('PASS: every reproduced application Yamada record is exactly unchanged relative to Latest_Workplace.')


In [ ]:
physics = [row for row in optimized if row['application'] == 'physics']
mathematics = [row for row in optimized if row['application'] == 'mathematics']
print(f'Physics cases: {len(physics)}')
for row in physics:
    print(f"{row['case']:14s} gamma={row['gamma']:<4} V={row['nodes']:<3} E={row['edges']:<3} crossings={row['crossings']:<2} Yamada={row['yamada']}")
print(f'\nMathematics cases: {len(mathematics)}')
for row in mathematics:
    print(row['case'], row.get('yamada', row.get('yamada_negami')))


## Interpretation

A pass means the current optimized implementation reproduces the `Latest_Workplace` application-level Yamada outputs exactly for all regression cases. The historical branch dependency is intentional and isolated to this correctness-regression notebook/helper.
